In [1]:
PREDICTION_INPUT_LENGTH = 30
PREDICTION_OUTPUT_LENGTH = 7

In [2]:
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import sys
import time

In [3]:
model_url = "https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-model/model_appl_trained.keras"
model_path = keras.utils.get_file("model_appl_trained.keras", model_url)

In [4]:
import requests
import json

json_url = "https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/indexes/index.json"
response = requests.get(json_url)
response.raise_for_status() # Raise an exception for HTTP errors
json_data = response.json()

print(json.dumps(json_data[:5], indent=2))

[
  {
    "ticker": "AAL",
    "filename": "AAL-NASDAQ.csv",
    "length": 4333,
    "exchange": "nasdaq"
  },
  {
    "ticker": "AAME",
    "filename": "AAME-NASDAQ.csv",
    "length": 10778,
    "exchange": "nasdaq"
  },
  {
    "ticker": "AAOI",
    "filename": "AAOI-NASDAQ.csv",
    "length": 2320,
    "exchange": "nasdaq"
  },
  {
    "ticker": "AAON",
    "filename": "AAON-NASDAQ.csv",
    "length": 7553,
    "exchange": "nasdaq"
  },
  {
    "ticker": "AAPL",
    "filename": "AAPL-NASDAQ.csv",
    "length": 10590,
    "exchange": "nasdaq"
  }
]


In [5]:
keras.backend.clear_session()

# Load the Keras model
loaded_model = tf.keras.models.load_model(model_path)

print("Model loaded successfully!")
loaded_model.summary()

Model loaded successfully!


Model: "Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 100)        │        45,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 100)        │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 7, 100)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 7, 100)         │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 100)         │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 7, 100)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 7, 5)           │           505 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 623,117 (2.38 MB)

 Trainable params: 207,505 (810.57 KB)

 Non-trainable params: 600 (2.34 KB)

 Optimizer params: 415,012 (1.58 MB)

In [6]:
halfsplit_points = [0.005, 0.02]
split_points = [-i for i in halfsplit_points][::-1] + halfsplit_points
print(split_points)
thresholds = np.array(split_points)
n_classes = len(split_points) + 1
bucket_ends = ['-inf'] + [f'{i:.3f}' for i in split_points] + ['inf']
bucket_names = [f'{bucket_ends[i]} -> {bucket_ends[i + 1]}' for i in range(n_classes)]
print(bucket_names)

cols_to_scale = ['volume', 'year', 'days', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'weekday_sin', 'weekday_cos']
scale_factor = 0.001

[-0.02, -0.005, 0.005, 0.02]
['-inf -> -0.020', '-0.020 -> -0.005', '-0.005 -> 0.005', '0.005 -> 0.020', '0.020 -> inf']


In [7]:
df_indexer = pd.DataFrame(json_data)
display(df_indexer.head())

,ticker,filename,length,exchange
0,AAL,AAL-NASDAQ.csv,4333,nasdaq
1,AAME,AAME-NASDAQ.csv,10778,nasdaq
2,AAOI,AAOI-NASDAQ.csv,2320,nasdaq
3,AAON,AAON-NASDAQ.csv,7553,nasdaq
4,AAPL,AAPL-NASDAQ.csv,10590,nasdaq


In [8]:
def get_top_n_companies_by_length(exchange_name, n, noise=0.0):
  """
  Returns a list of n companies from the specified exchange with the highest 'length' values.
  The 'noise' parameter introduces randomness into the selection process.

  Args:
    exchange_name (str): The name of the exchange (e.g., 'nasdaq').
    n (int): The number of top companies to return.
    noise (float): A non-negative value between 0 and 1 (inclusive) representing
                   the level of randomness.
                   - If 0, selection is purely based on 'length'.
                   - If 1, selection is purely random.
                   - Values between 0 and 1 blend length-based and random selection.

  Returns:
    list: A list of dictionaries, where each dictionary represents a company
          selected based on the weighted scores.
  """
  # Filter the DataFrame for the specified exchange
  exchange_df = df_indexer[df_indexer['exchange'] == exchange_name].copy()
  if noise < 0 or noise > 1:
      # Clamp noise to be within [0, 1]
      noise = max(0.0, min(noise, 1.0))
  if noise == 0:
    # Purely length-based sorting
    top_n_companies = exchange_df.sort_values(by='length', ascending=False).head(n)
  else:
    # Normalize length values to a 0-1 range
    min_len = exchange_df['length'].min()
    max_len = exchange_df['length'].max()
    if max_len == min_len: # Handle cases where all lengths are the same
        normalized_length = pd.Series(0.5, index=exchange_df.index)
    else:
        normalized_length = (exchange_df['length'] - min_len) / (max_len - min_len)
    # Generate random scores (uniform distribution between 0 and 1)
    random_scores = np.random.rand(exchange_df.shape[0])
    # Combine normalized length and random scores based on noise
    # A higher 'noise' value gives more weight to the random score.
    exchange_df['selection_score'] = (1 - noise) * normalized_length + noise * random_scores
    # Sort by the combined selection score
    top_n_companies = exchange_df.sort_values(by='selection_score', ascending=False).head(n)
  # Convert the result to a list of dictionaries
  return top_n_companies.to_dict(orient='records')

In [9]:
exchange_counts = df_indexer['exchange'].value_counts()
available_exchanges = exchange_counts.index.tolist()

print("Available Exchanges (by popularity descending):")
for exchange in available_exchanges:
    print(f"- {exchange} ({exchange_counts[exchange]} companies)")

Available Exchanges (by popularity descending):
- nasdaq (1564 companies)
- upcomindex (871 companies)
- vnindex (416 companies)
- hnxindex (342 companies)


# Task 3: Buying/Selling Signal
Since we are day-trading, our strategy is simple. Based on the 1-day prediction, if it goes up then we buy now and sell later, and vice versa.

In [10]:
# Model constants
example_df = pd.read_csv(f'https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/{json_data[0]["filename"]}')
feature_count = len(example_df.keys())
window_input = PREDICTION_INPUT_LENGTH
window_output = PREDICTION_OUTPUT_LENGTH
shape_input = (window_input, feature_count)
shape_output = (window_output, n_classes)
print(f"Input shape: {shape_input}")
print(f"Output shape: {shape_output}")

Input shape: (30, 13)
Output shape: (7, 5)


In [11]:
# Trading algorithm
confidence_threshold = 0.3 # Increse to reduce risk, decrease to increase possible profit
max_trade = 1 # Linearly scales with our result

# Trading constants
result_returns = {0: -0.02, 1: -0.005, 2: 0.0, 3: 0.005, 4: 0.02}

In [12]:
companies_per_exchange = 20
trading_attempts = 500
tttotal_spent = 0.0
tttotal_profit = 0.0
print("We'll use our buy/sell strategy to trade with:")
for exchange in available_exchanges:
    top_companies = get_top_n_companies_by_length(exchange, companies_per_exchange, noise=0.2)
    print(f"\tThe top {companies_per_exchange} historical companies on {exchange}:")
    ttotal_spent = 0.0
    ttotal_profit = 0.0
    for company in top_companies:
        print(f'\t\tTrading with {company['ticker']} on {exchange} for the last {trading_attempts}/{company['length']} days...')
        data_url = f'https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/{company["filename"]}'
        data_df = pd.read_csv(data_url)
        data_df = data_df.tail(trading_attempts + window_input + window_output)
        data_df = data_df.assign(**{c: lambda d, c=c: d[c] * scale_factor for c in cols_to_scale}).clip(-1, 1)
        X_data = []
        Y_data = []
        for i in range(len(data_df) - window_input - window_output):
            data_input = data_df.iloc[i : i + window_input].values
            raw_y = data_df.iloc[i + window_input : i + window_input + window_output, 2].values
            bucketed = np.digitize(raw_y, thresholds)
            data_output = keras.utils.to_categorical(bucketed, num_classes=n_classes)
            X_data.append(data_input)
            Y_data.append(data_output)
        X_data = np.array(X_data)
        Y_data = np.array(Y_data)
        Y_data_1d = Y_data[:, 0, :]
        Y_pred = loaded_model.predict(X_data, verbose='false')
        Y_pred_1d = Y_pred[:, 0, :]
        total_spent = 0.0
        total_profit = 0.0
        for i in range(len(Y_data_1d)):
            conf = np.max(Y_pred_1d[i])
            pred_pos = np.argmax(Y_pred_1d[i])
            actual_pos = np.argmax(Y_data_1d[i])
            # Check threshold and skip neutral (pos 2)
            if conf > confidence_threshold and pred_pos != 2:
                bet_amount = conf * max_trade
                total_spent += bet_amount
                outcome_return = result_returns[actual_pos]
                # Short (0, 1) vs Long (3, 4) logic
                if pred_pos in [0, 1]:
                    # Inverse return for short positions
                    profit = -1 * outcome_return * bet_amount
                else:
                    # Direct return for long positions
                    profit = outcome_return * bet_amount
                total_profit += profit
        print(f'\t\t-> We spent ${total_spent} and got ${total_profit + total_spent} back, resulting in ${total_profit} of profits.')
        ttotal_spent += total_spent
        ttotal_profit += total_profit
    print(f'\t-> On {exchange}, we spent ${ttotal_spent} and got ${ttotal_profit + ttotal_spent} back, resulting in ${ttotal_profit} of profits.')
    tttotal_spent += ttotal_spent
    tttotal_profit += ttotal_profit
print(f'-> In total, we spent ${tttotal_spent} and got ${tttotal_profit + tttotal_spent} back, resulting in ${tttotal_profit} of profits.')

We'll use our buy/sell strategy to trade with:
	The top 20 historical companies on nasdaq:
		Trading with SHLM on nasdaq for the last 500/12593 days...
		-> We spent $189.6045379638672 and got $191.37969970703125 back, resulting in $1.775160551071167 of profits.
		Trading with DIOD on nasdaq for the last 500/12564 days...
		-> We spent $265.19873046875 and got $268.4409484863281 back, resulting in $3.2422287464141846 of profits.
		Trading with ALOG on nasdaq for the last 500/12597 days...
		-> We spent $176.22625732421875 and got $177.4849090576172 back, resulting in $1.2586497068405151 of profits.
		Trading with SGC on nasdaq for the last 500/12564 days...
		-> We spent $258.6112365722656 and got $261.706298828125 back, resulting in $3.0950498580932617 of profits.
		Trading with TXN on nasdaq for the last 500/12744 days...
		-> We spent $222.5488739013672 and got $224.34654235839844 back, resulting in $1.7976735830307007 of profits.
		Trading with PHI on nasdaq for the last 500/12514 

In [35]:
print(f'Profit margin: {tttotal_profit / tttotal_spent * 100}%')

Profit margin: 1.0975801944732666%


Keep in mind that our gains of 0.5% or 2% is the "minimum" gain of these scenarios, the actual value is much higher

# Task 4:
## Task 4.1: Profitable stock selection
As we can see from Task 3, no matter the company, we can always make a profit. Therefore, no selection is needed.
## Task 4.2: Risk management
Our strategy to evaluate the risk of companies is to see which companies does our model (currently only trained on NASDAQ-AAPL data) performs the best.

In [14]:
companies_per_exchange = 100
testing_attempts = 100
print(f"Risk Assessment - 7-days prediction")
result_risk_7d = []
for exchange in available_exchanges:
    top_companies = get_top_n_companies_by_length(exchange, companies_per_exchange, noise=0.2)
    print(f"\tThe top {companies_per_exchange} historical companies on {exchange}:")
    start_time = time.time()
    for ci, company in enumerate(top_companies):
        # print(f'\t\tVerifying our model with {company['ticker']} on {exchange} for the last {testing_attempts}/{company['length']} days...')
        data_url = f'https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/{company["filename"]}'
        data_df = pd.read_csv(data_url)
        data_df = data_df.tail(testing_attempts + window_input + window_output)
        data_df = data_df.assign(**{c: lambda d, c=c: d[c] * scale_factor for c in cols_to_scale}).clip(-1, 1)
        X_data = []
        Y_data = []
        for i in range(len(data_df) - window_input - window_output):
            data_input = data_df.iloc[i : i + window_input].values
            raw_y = data_df.iloc[i + window_input : i + window_input + window_output, 2].values
            bucketed = np.digitize(raw_y, thresholds)
            data_output = keras.utils.to_categorical(bucketed, num_classes=n_classes)
            X_data.append(data_input)
            Y_data.append(data_output)
        X_data = np.array(X_data)
        Y_data = np.array(Y_data)
        # Y_data_1d = Y_data[:, 0, :]
        Y_pred = loaded_model.predict(X_data, verbose='false')
        # Y_pred_1d = Y_pred[:, 0, :]
        Y_value = np.argmax(Y_data, axis=-1).flatten()
        Y_pred_value = np.argmax(Y_pred, axis=-1).flatten()
        accuracy = sklearn.metrics.accuracy_score(Y_value, Y_pred_value)
        result_risk_7d.append({
            'exchange': exchange,
            'ticker': company['ticker'],
            'accuracy': accuracy,
            'fullname': f'{exchange}-{company["ticker"]}'
        })
        elapsed_time = time.time() - start_time
        mins, secs = divmod(int(elapsed_time), 60)
        sys.stdout.write(f"\r\tProcessing item {ci + 1}/{companies_per_exchange} | {((ci + 1)/companies_per_exchange * 100):.1f}% | Time elapsed: {mins:02}:{secs:02}    ")
        sys.stdout.flush()
    print("\n")
df_risk_7d = pd.DataFrame(result_risk_7d)

Risk Assessment - 7-days prediction
	The top 100 historical companies on nasdaq:
	Processing item 100/100 | 100.0% | Time elapsed: 01:39    

	The top 100 historical companies on upcomindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:20    

	The top 100 historical companies on vnindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:25    

	The top 100 historical companies on hnxindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:22    



In [17]:
amount = 20
print(f'Top {amount} companies with lowest 7-days risk')
display(df_risk_7d.sort_values(by='accuracy', ascending=False).head(amount))

Top 20 companies with lowest 7-days risk


,exchange,ticker,accuracy,fullname
270,vnindex,MCG,0.408571,vnindex-MCG
396,hnxindex,L18,0.405714,hnxindex-L18
290,vnindex,TCR,0.401429,vnindex-TCR
281,vnindex,DIG,0.395714,vnindex-DIG
392,hnxindex,TNG,0.395714,hnxindex-TNG
76,nasdaq,VALU,0.391429,nasdaq-VALU
297,vnindex,DXG,0.390000,vnindex-DXG
65,nasdaq,FULT,0.390000,nasdaq-FULT
93,nasdaq,CLFD,0.388571,nasdaq-CLFD
349,hnxindex,PVC,0.384286,hnxindex-PVC


In [18]:
companies_per_exchange = 100
testing_attempts = 100
print(f"Risk Assessment - 1-days prediction")
result_risk_1d = []
for exchange in available_exchanges:
    top_companies = get_top_n_companies_by_length(exchange, companies_per_exchange, noise=0.2)
    print(f"\tThe top {companies_per_exchange} historical companies on {exchange}:")
    start_time = time.time()
    for ci, company in enumerate(top_companies):
        # print(f'\t\tVerifying our model with {company['ticker']} on {exchange} for the last {testing_attempts}/{company['length']} days...')
        data_url = f'https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/{company["filename"]}'
        data_df = pd.read_csv(data_url)
        data_df = data_df.tail(testing_attempts + window_input + window_output)
        data_df = data_df.assign(**{c: lambda d, c=c: d[c] * scale_factor for c in cols_to_scale}).clip(-1, 1)
        X_data = []
        Y_data = []
        for i in range(len(data_df) - window_input - window_output):
            data_input = data_df.iloc[i : i + window_input].values
            raw_y = data_df.iloc[i + window_input : i + window_input + window_output, 2].values
            bucketed = np.digitize(raw_y, thresholds)
            data_output = keras.utils.to_categorical(bucketed, num_classes=n_classes)
            X_data.append(data_input)
            Y_data.append(data_output)
        X_data = np.array(X_data)
        Y_data = np.array(Y_data)
        Y_data_1d = Y_data[:, 0, :]
        Y_pred = loaded_model.predict(X_data, verbose='false')
        Y_pred_1d = Y_pred[:, 0, :]
        Y_value = np.argmax(Y_data_1d, axis=-1).flatten()
        Y_pred_value = np.argmax(Y_pred_1d, axis=-1).flatten()
        accuracy = sklearn.metrics.accuracy_score(Y_value, Y_pred_value)
        result_risk_1d.append({
            'exchange': exchange,
            'ticker': company['ticker'],
            'accuracy': accuracy,
            'fullname': f'{exchange}-{company["ticker"]}'
        })
        elapsed_time = time.time() - start_time
        mins, secs = divmod(int(elapsed_time), 60)
        sys.stdout.write(f"\r\tProcessing item {ci + 1}/{companies_per_exchange} | {((ci + 1)/companies_per_exchange * 100):.1f}% | Time elapsed: {mins:02}:{secs:02}    ")
        sys.stdout.flush()
    print("\n")
df_risk_1d = pd.DataFrame(result_risk_1d)

Risk Assessment - 1-days prediction
	The top 100 historical companies on nasdaq:
	Processing item 100/100 | 100.0% | Time elapsed: 01:51    

	The top 100 historical companies on upcomindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:19    

	The top 100 historical companies on vnindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:21    

	The top 100 historical companies on hnxindex:
	Processing item 100/100 | 100.0% | Time elapsed: 01:18    



In [19]:
amount = 20
print(f'Top {amount} companies with lowest 1-days risk')
display(df_risk_1d.sort_values(by='accuracy', ascending=False).head(amount))

Top 20 companies with lowest 1-days risk


,exchange,ticker,accuracy,fullname
334,hnxindex,BLF,1.00,hnxindex-BLF
340,hnxindex,TBX,1.00,hnxindex-TBX
158,upcomindex,VCT,1.00,upcomindex-VCT
388,hnxindex,CJC,1.00,hnxindex-CJC
332,hnxindex,SGD,0.96,hnxindex-SGD
398,hnxindex,L35,0.92,hnxindex-L35
162,upcomindex,TVG,0.91,upcomindex-TVG
177,upcomindex,PTP,0.91,upcomindex-PTP
310,hnxindex,SDC,0.91,hnxindex-SDC
176,upcomindex,PTG,0.90,upcomindex-PTG


## Task 4.3: Portfolio composition
As stated, with our trading model, we cannot reliably create an investment portfolio. However, from our findings, we can optimize our strategy for day-trading purposes

In [31]:
# Let's trade with only companies that we know our model performs the best on, excluding the "too accurate" ones since it'll probably be zero movement all the time
# Even though we are evaluating on the same "days" that we tested on, the behavior should be fine
max_companies_to_trade = 20
companies_list = filter(lambda x: 0.75 > x['accuracy'] < 0.85, result_risk_1d)
companies_list = sorted(companies_list, key=lambda x: x['accuracy'], reverse=True)[:max_companies_to_trade]
for company in companies_list:
  company['filename'] = f"{company['ticker']}-{company['exchange'].upper()}.csv"

print(companies_list[:5])

[{'exchange': 'nasdaq', 'ticker': 'XCRA', 'accuracy': 0.73, 'fullname': 'nasdaq-XCRA', 'filename': 'XCRA-NASDAQ.csv'}, {'exchange': 'nasdaq', 'ticker': 'ESIO', 'accuracy': 0.73, 'fullname': 'nasdaq-ESIO', 'filename': 'ESIO-NASDAQ.csv'}, {'exchange': 'upcomindex', 'ticker': 'VTS', 'accuracy': 0.73, 'fullname': 'upcomindex-VTS', 'filename': 'VTS-UPCOMINDEX.csv'}, {'exchange': 'nasdaq', 'ticker': 'SHLM', 'accuracy': 0.72, 'fullname': 'nasdaq-SHLM', 'filename': 'SHLM-NASDAQ.csv'}, {'exchange': 'upcomindex', 'ticker': 'BT6', 'accuracy': 0.72, 'fullname': 'upcomindex-BT6', 'filename': 'BT6-UPCOMINDEX.csv'}]


In [33]:
trading_attempts = 100
ttotal_spent = 0.0
ttotal_profit = 0.0
print("We'll use our buy/sell strategy to trade with:")
for company in companies_list:
    print(f'\tTrading with {company['ticker']} on {company['exchange']} for the last {trading_attempts} days...')
    data_url = f'https://github.com/malego290704/fuv-deeplearning-finalproject-v1/raw/refs/heads/main/v1-data-normalized/{company["filename"]}'
    data_df = pd.read_csv(data_url)
    data_df = data_df.tail(trading_attempts + window_input + window_output)
    data_df = data_df.assign(**{c: lambda d, c=c: d[c] * scale_factor for c in cols_to_scale}).clip(-1, 1)
    X_data = []
    Y_data = []
    for i in range(len(data_df) - window_input - window_output):
        data_input = data_df.iloc[i : i + window_input].values
        raw_y = data_df.iloc[i + window_input : i + window_input + window_output, 2].values
        bucketed = np.digitize(raw_y, thresholds)
        data_output = keras.utils.to_categorical(bucketed, num_classes=n_classes)
        X_data.append(data_input)
        Y_data.append(data_output)
    X_data = np.array(X_data)
    Y_data = np.array(Y_data)
    Y_data_1d = Y_data[:, 0, :]
    Y_pred = loaded_model.predict(X_data, verbose='false')
    Y_pred_1d = Y_pred[:, 0, :]
    total_spent = 0.0
    total_profit = 0.0
    for i in range(len(Y_data_1d)):
        conf = np.max(Y_pred_1d[i])
        pred_pos = np.argmax(Y_pred_1d[i])
        actual_pos = np.argmax(Y_data_1d[i])
        # Check threshold and skip neutral (pos 2)
        if conf > confidence_threshold and pred_pos != 2:
            bet_amount = conf * max_trade
            total_spent += bet_amount
            outcome_return = result_returns[actual_pos]
            # Short (0, 1) vs Long (3, 4) logic
            if pred_pos in [0, 1]:
                # Inverse return for short positions
                profit = -1 * outcome_return * bet_amount
            else:
                # Direct return for long positions
                profit = outcome_return * bet_amount
            total_profit += profit
    print(f'\t-> We spent ${total_spent} and got ${total_profit + total_spent} back, resulting in ${total_profit} of profits.')
    ttotal_spent += total_spent
    ttotal_profit += total_profit
print(f'-> In total, we spent ${ttotal_spent} and got ${ttotal_profit + ttotal_spent} back, resulting in ${ttotal_profit} of profits.')

We'll use our buy/sell strategy to trade with:
	Trading with XCRA on nasdaq for the last 100 days...
	-> We spent $39.003074645996094 and got $39.2713623046875 back, resulting in $0.2682858407497406 of profits.
	Trading with ESIO on nasdaq for the last 100 days...
	-> We spent $29.443899154663086 and got $29.726778030395508 back, resulting in $0.2828781008720398 of profits.
	Trading with VTS on upcomindex for the last 100 days...
	-> We spent $6.659021377563477 and got $6.651930332183838 back, resulting in $-0.007090926170349121 of profits.
	Trading with SHLM on nasdaq for the last 100 days...
	-> We spent $19.246479034423828 and got $19.296642303466797 back, resulting in $0.05016312748193741 of profits.
	Trading with BT6 on upcomindex for the last 100 days...
	-> We spent $9.809926986694336 and got $9.876008033752441 back, resulting in $0.0660809800028801 of profits.
	Trading with DIC on upcomindex for the last 100 days...
	-> We spent $14.297929763793945 and got $14.492295265197754 b

In [36]:
print(f'Profit margin: {ttotal_profit / ttotal_spent * 100}%')

Profit margin: 1.3192996978759766%


We have an increase in our profit margin by applying the risk analysis for our "trading portfolio" stocks!